# Manhattan Experiment: 4-TX Challenge Scenario

**Scene:** Manhattan — four transmitters with distinct organic coverage zones  
**Frequency:** 3.5 GHz

The challenge scenario stresses gradient-based optimization with:
- Higher TX count → stronger inter-cell interference
- Diverse zone geometries → varied gradient landscapes
- Dense urban clutter → more coverage voids

| TX | Zone Shape | Description |
|---|---|---|
| gnb1 | Wide strip | Main avenue corridor |
| gnb2 | T-shape | Intersection: main + side street |
| gnb3 | Irregular polygon | Park or open plaza |
| gnb4 | L-shape | Narrow alley + side street corner |

> **Before running:** Set `best_n` below to the sample count from the Duke ablation.

In [ ]:
# ── CONFIGURATION ─────────────────────────────────────────────────────────────
SCENE_XML_PATH = "../scene/scenes/Manhattan/scene.xml"
CARRIER_HZ     = 3.5e9

# Sample count from Duke ablation — set this before running
best_n = 256    # ← set from duke_experiment.ipynb

# Transmitters — adjust building IDs to valid buildings in the Manhattan scene
TX_DEFS = [
    {"name": "gnb1", "building_id":  7, "height_m": 10.0},
    {"name": "gnb2", "building_id": 15, "height_m": 10.0},
    {"name": "gnb3", "building_id": 23, "height_m": 10.0},
    {"name": "gnb4", "building_id": 31, "height_m": 10.0},
]

# ── ZONE CONFIGURATION ──────────────────────────────────────────────────────
# Coordinates are scene-local meters. Run the visualization cell to verify.
#
# Zone 1 — Wide corridor strip (main avenue, slightly skewed)
ZONE1_VERTICES = [
    (-400.0,  20.0),
    ( 100.0,  30.0),
    ( 100.0, -30.0),
    (-400.0, -40.0),
]

# Zone 2 — T-shape (horizontal bar + vertical stem)
# ┌──────────────┐
# │              │
# └────┬────┬────┘
#      │    │
#      └────┘
ZONE2_VERTICES = [
    (-150.0,  300.0),
    ( 150.0,  300.0),
    ( 150.0,  150.0),
    (  60.0,  150.0),
    (  60.0,   50.0),
    ( -60.0,   50.0),
    ( -60.0,  150.0),
    (-150.0,  150.0),
]

# Zone 3 — Irregular park polygon
ZONE3_VERTICES = [
    ( 200.0, -100.0),
    ( 380.0, -130.0),
    ( 430.0,   20.0),
    ( 350.0,  120.0),
    ( 180.0,   80.0),
    ( 150.0,  -30.0),
]

# Zone 4 — Narrow L-shape (alley corner)
ZONE4_VERTICES = [
    (-200.0, -200.0),
    ( -80.0, -200.0),
    ( -80.0, -320.0),
    (-120.0, -320.0),
    (-120.0, -250.0),
    (-200.0, -250.0),
]

ZONE_VERTICES = [ZONE1_VERTICES, ZONE2_VERTICES, ZONE3_VERTICES, ZONE4_VERTICES]
ZONE_NAMES    = ["Wide strip", "T-shape", "Irregular park", "L-shape alley"]
ZONE_COLORS   = ["steelblue", "darkorange", "darkgreen", "mediumpurple"]
ZONE_CMAPS    = ["Blues", "Oranges", "Greens", "Purples"]

MAP_CONFIG = {
    'center':        [0.0, 0.0, 0.0],
    'size':          [1400, 1400],
    'cell_size':     (0.5, 0.5),
    'ground_height': 0.0,
}

# Experiment
NOISE_POWER    = 1e-10
JITTER_SEED    = 42
JITTER_MAG     = 1e-4
NUM_ITERATIONS = 50
BASELINES      = ["grad_full_rejection", "grad_full_triangulated", "grad_proportional"]
METRICS        = ["rsrp_mean_dbm", "rsrp_p10_dbm", "sir_median_db", "sir_p10_db", "coverage_fraction"]
OUTPUT_PATH    = "../scripts/report/manhattan_experiment_results.json"

In [ ]:
import sys, os
sys.path.append(os.path.abspath('../src'))
import warnings; warnings.filterwarnings("ignore")
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as MplPolygon
import mitsuba as mi

try:
    import sionna.rt
except ImportError:
    os.system("pip install sionna-rt")
    import sionna.rt

from sionna.rt import load_scene, AntennaArray
from sionna.rt.antenna_pattern import antenna_pattern_registry

from scene_parser import extract_building_info
from tx_placement import TxPlacement
from boresight_pathsolver import create_zone_mask
from angle_utils import compute_initial_angles_from_position, azimuth_elevation_to_yaw_pitch
from multi_tx_optimizer import TxConfig
from experiment_runner import (
    ExperimentConfig, run_experiment_suite,
    compare_all_results, get_distribution_data,
    plot_cdf, plot_loss_curves, plot_metric_bars,
)

scene = load_scene(SCENE_XML_PATH)
scene.frequency = CARRIER_HZ

single_el = np.array([[0.0, 0.0, 0.0]])
scene.tx_array = AntennaArray(
    antenna_pattern=antenna_pattern_registry.get("tr38901")(polarization="V"),
    normalized_positions=single_el.T,
)
scene.rx_array = AntennaArray(
    antenna_pattern=antenna_pattern_registry.get("iso")(polarization="V"),
    normalized_positions=single_el.T,
)
for rm in scene.radio_materials.values():
    rm.scattering_coefficient = 0.4

building_info = extract_building_info(SCENE_XML_PATH, verbose=False)
print(f"Scene loaded  |  {CARRIER_HZ/1e9:.1f} GHz  |  {len(building_info)} buildings")

In [ ]:
# ── ZONE MASKS (all 4) ───────────────────────────────────────────────────────
zone_params_list = [{"vertices": v} for v in ZONE_VERTICES]
zone_masks_list  = []
zone_stats_list  = []

for i, (zp, zname) in enumerate(zip(zone_params_list, ZONE_NAMES)):
    mask, look_at, stats = create_zone_mask(
        map_config=MAP_CONFIG,
        zone_type='polygon',
        zone_params=zp,
        target_height=1.5,
        scene_xml_path=SCENE_XML_PATH,
        exclude_buildings=True,
    )
    zone_masks_list.append(mask)
    zone_stats_list.append(stats)
    print(f"Zone {i+1} ({zname}): {stats['num_cells']} cells  centroid={stats['centroid_xy']}")

zone_masks = {td["name"]: zone_masks_list[i] for i, td in enumerate(TX_DEFS)}

In [ ]:
# ── TX1 PLACEMENT ─────────────────────────────────────────────────────────────
td = TX_DEFS[0]
tp1 = TxPlacement(scene, td["name"], SCENE_XML_PATH, td["building_id"], offset=td["height_m"])
tp1.set_rooftop_zone_facing(zone_stats_list[0]["centroid_xy"])
tx1_pos = scene.get(td["name"]).position.numpy().flatten().tolist()
print(f"{td['name']} at {tx1_pos}")

In [ ]:
# ── TX2 PLACEMENT ─────────────────────────────────────────────────────────────
td = TX_DEFS[1]
tp2 = TxPlacement(scene, td["name"], SCENE_XML_PATH, td["building_id"], offset=td["height_m"])
tp2.set_rooftop_zone_facing(zone_stats_list[1]["centroid_xy"])
tx2_pos = scene.get(td["name"]).position.numpy().flatten().tolist()
print(f"{td['name']} at {tx2_pos}")

In [ ]:
# ── TX3 PLACEMENT ─────────────────────────────────────────────────────────────
td = TX_DEFS[2]
tp3 = TxPlacement(scene, td["name"], SCENE_XML_PATH, td["building_id"], offset=td["height_m"])
tp3.set_rooftop_zone_facing(zone_stats_list[2]["centroid_xy"])
tx3_pos = scene.get(td["name"]).position.numpy().flatten().tolist()
print(f"{td['name']} at {tx3_pos}")

In [ ]:
# ── TX4 PLACEMENT ─────────────────────────────────────────────────────────────
td = TX_DEFS[3]
tp4 = TxPlacement(scene, td["name"], SCENE_XML_PATH, td["building_id"], offset=td["height_m"])
tp4.set_rooftop_zone_facing(zone_stats_list[3]["centroid_xy"])
tx4_pos = scene.get(td["name"]).position.numpy().flatten().tolist()
print(f"{td['name']} at {tx4_pos}")

tx_positions = [tx1_pos, tx2_pos, tx3_pos, tx4_pos]

In [ ]:
# ── INITIAL JITTER ───────────────────────────────────────────────────────────
rng = np.random.default_rng(JITTER_SEED)
_init_state = {}   # {tx_name: (pos, yaw, pitch)}

for td, tx_pos, stats in zip(TX_DEFS, tx_positions, zone_stats_list):
    base_az, base_el = compute_initial_angles_from_position(tx_pos, stats["look_at_xyz"])
    init_az = base_az + float(rng.uniform(-JITTER_MAG, JITTER_MAG))
    init_el = base_el + float(rng.uniform(-JITTER_MAG, JITTER_MAG))
    yaw_r, pitch_r = azimuth_elevation_to_yaw_pitch(init_az, init_el)
    tx = scene.get(td["name"])
    tx.orientation = mi.Point3f(yaw_r, pitch_r, 0.0)
    _init_state[td["name"]] = (tx_pos[:], yaw_r, pitch_r)
    print(f"{td['name']}  Az={init_az:.6f}°  El={init_el:.6f}°")

In [ ]:
# ── ZONE VISUALIZATION ───────────────────────────────────────────────────────
_cx, _cy, _ = MAP_CONFIG['center']
_w, _h = MAP_CONFIG['size']
extent = [_cx - _w/2, _cx + _w/2, _cy - _h/2, _cy + _h/2]

fig, ax = plt.subplots(figsize=(10, 10))
markers = ['^', 's', 'D', 'P']

for i, (td, mask, stats) in enumerate(zip(TX_DEFS, zone_masks_list, zone_stats_list)):
    ax.imshow(np.ma.masked_where(mask == 0, mask),
              origin='lower', extent=extent,
              cmap=ZONE_CMAPS[i], vmin=0, vmax=1, alpha=0.4)
    pos = tx_positions[i]
    ax.plot(pos[0], pos[1], markers[i], color=ZONE_COLORS[i], markersize=14,
            markeredgecolor='black', label=f"{td['name']} — {ZONE_NAMES[i]}", zorder=5)
    ax.plot(*stats['centroid_xy'], 'o', color=ZONE_COLORS[i], markersize=8,
            markeredgecolor='k', zorder=5)

for bdata in building_info.values():
    verts = bdata['vertices'][:, :2]
    ax.add_patch(MplPolygon(verts, closed=True, facecolor='gray',
                            edgecolor='black', linewidth=0.8, alpha=0.35))

ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)')
ax.set_title('Manhattan — 4 TX, Organic Coverage Zones (3.5 GHz)')
ax.legend(loc='upper right'); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# ── CONVERGENCE HELPER ───────────────────────────────────────────────────────
def check_convergence(suite_output, baseline_id, tx_name, tol=1e-4):
    entry = suite_output["results"].get(baseline_id, {})
    opt   = entry.get("optimizer_result")
    if not isinstance(opt, dict):
        return {"status": "error"}
    tx_res  = opt.get(tx_name, {})
    az_hist = tx_res.get("az_history", [])
    el_hist = tx_res.get("el_history", [])
    if len(az_hist) > 1:
        for i in range(1, len(az_hist)):
            if abs(az_hist[i] - az_hist[i-1]) < tol and abs(el_hist[i] - el_hist[i-1]) < tol:
                return {"status": "converged", "iteration": i}
        return {"status": "no convergence", "iterations": len(az_hist)}
    init_angles = suite_output["metadata"]["initial_angles"].get(tx_name, [None, None])
    best_angles = tx_res.get("best_angles", [None, None])
    if None in list(init_angles) + list(best_angles):
        return {"status": "error"}
    d_az = abs(best_angles[0] - init_angles[0])
    d_el = abs(best_angles[1] - init_angles[1])
    label = "no movement" if d_az < tol and d_el < tol else "displaced"
    return {"status": label, "Δaz_deg": round(d_az, 4), "Δel_deg": round(d_el, 4)}

In [ ]:
# ── RUN SUITE ────────────────────────────────────────────────────────────────
tx_configs = [
    TxConfig(
        name=td["name"],
        building_id=td["building_id"],
        zone_params=zp,
        tx_height_offset=td["height_m"],
        num_sample_points=best_n,
    )
    for td, zp in zip(TX_DEFS, zone_params_list)
]

exp_config = ExperimentConfig(
    baselines=BASELINES,
    noise_power=NOISE_POWER,
    num_iterations=NUM_ITERATIONS,
    output_path=OUTPUT_PATH,
)

suite_output = run_experiment_suite(
    scene=scene,
    tx_configs=tx_configs,
    map_config=MAP_CONFIG,
    scene_xml_path=SCENE_XML_PATH,
    zone_masks=zone_masks,
    exp_config=exp_config,
)

In [ ]:
# ── CONVERGENCE ANALYSIS ─────────────────────────────────────────────────────
tx_names = [td["name"] for td in TX_DEFS]

print(f"\n{'─'*65}")
print(f"  Convergence Analysis  (tol = {JITTER_MAG:.0e} deg)")
print(f"{'─'*65}")
for bid in BASELINES:
    print(f"  {bid}")
    for tx_name in tx_names:
        conv = check_convergence(suite_output, bid, tx_name, tol=JITTER_MAG)
        print(f"    {tx_name:<8s}  {conv}")

In [ ]:
# ── RESULTS TABLE ────────────────────────────────────────────────────────────
df = compare_all_results(suite_output, metrics=METRICS)

In [ ]:
# ── PER-TX BREAKDOWN ─────────────────────────────────────────────────────────
print(f"\n{'═'*65}")
print("  Per-TX Optimized Stats")
print(f"{'═'*65}")
for bid in BASELINES:
    print(f"\n  {bid}")
    cstats = suite_output["results"][bid].get("comparison_stats", {})
    if not isinstance(cstats, dict):
        print("    [FAILED]")
        continue
    for tx_name in tx_names:
        opt = cstats.get(tx_name, {}).get("optimized", {})
        print(f"    {tx_name:<8s}"
              f"  RSRP_mean={opt.get('rsrp_mean_dbm', float('nan')):.2f} dBm"
              f"  SIR_med={opt.get('sir_median_db', float('nan')):.2f} dB"
              f"  cov={opt.get('coverage_fraction', float('nan')):.3f}")

In [ ]:
# ── VISUALIZATION ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

plot_metric_bars(
    suite_output,
    metrics=("rsrp_mean_dbm", "rsrp_p10_dbm", "sir_median_db", "sir_p10_db", "coverage_fraction"),
    ax=axes[0],
)
axes[0].set_title("Metric Comparison — All TXs (avg)")

plot_loss_curves(suite_output, ax=axes[1])
axes[1].set_title("Convergence Curves")

plot_cdf(suite_output, metric="rsrp_values_dbm", ax=axes[2])
axes[2].set_title("RSRP CDF")

plt.suptitle(f"Manhattan Experiment — 4 TX, Organic Zones, 3.5 GHz (n={best_n})", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── PER-TX CDF GRID ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for col_i, tx_name in enumerate(tx_names):
    for row_i, metric in enumerate(["rsrp_values_dbm", "sir_values_db"]):
        ax = axes[row_i][col_i]
        plot_cdf(suite_output, metric=metric, tx_name=tx_name, ax=ax)
        ax.set_title(f"{tx_name}\n{'RSRP' if row_i == 0 else 'SINR'}", fontsize=9)
        if col_i > 0:
            ax.set_ylabel("")

plt.suptitle("Per-TX CDF: RSRP (top) and SINR (bottom)", y=1.02)
plt.tight_layout()
plt.show()